## Planning:
- Load data
- Check variable and target
- Using CatBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
submit = pd.read_csv("data/sample_submission.csv")

In [3]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [4]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [5]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [6]:
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

## Basic data treatment

In [12]:
# change boolean to 1 and 0
for col in bool_cols:
    train[col] = train[col].astype(int)
    test[col] = test[col].astype(int)

## Model using CatBoost

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [10]:
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [22]:
X = train.drop(columns=["id","is_fraud"])
y = train["is_fraud"]
X_test = test.drop(columns=["id"])

In [34]:
SEED = 67
N_FOLDS = 5
TARGET = 'is_fraud'

In [24]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [25]:
def fit_catboost(X_tr, y_tr, X_va, y_va):
    model = CatBoostClassifier(
        loss_function='Logloss',
        eval_metric='AUC',
        iterations=1000,
        learning_rate=0.01,
        depth=7,
        l2_leaf_reg=1.0,
        random_seed=SEED,
        thread_count=-1,
        early_stopping_rounds=200,
        verbose=False
    )
    model.fit(X_tr, y_tr, 
              eval_set=[(X_va, y_va)], 
              cat_features = obj_cols,
              early_stopping_rounds = 200,
              verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [31]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_cb
    oof_df.to_csv(name_2,index=False)
    print('save:', name_2)

In [35]:
oof_cb, test_cb, auc_cb = run_cv(fit_catboost, 'CatBoost')

  fold 0: AUC = 0.94104
  fold 1: AUC = 0.94408
  fold 2: AUC = 0.94842
  fold 3: AUC = 0.94408
  fold 4: AUC = 0.93829
CatBoost | OOF AUC = 0.94313 (mean 0.94318 ± 0.00339)


In [37]:
save_sub(test_cb, oof_cb, f'sub/sub_catboost_{SEED}.csv', f'oof/oof_catboost_{SEED}.csv')

save: sub/sub_catboost_67.csv
save: oof/oof_catboost_67.csv
